# Word Embedding을 활용한 단어 게임

- Level 1. [Semantle] (https://semantle.com) 흉내내기
- Level 2. 한국어 
- Level 3. 단어 'A:B=C:? ?맟추기 word2vec 흉내내기 + Semantle 

In [ ]:
# print(model.wv.similarity('people', 'think'))   #similarity measure between the two words


In [4]:
import json

with open('corpus_text.json', 'r', encoding='utf-8') as f:
    corpus_text = json.load(f)

tokenized_corpus = [sentence.split() for sentence in corpus_text]

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,    #vocab size
    window=5,           #CBOW / Skip-gram: surrounding words count (pre & post look at 5) 
    min_count=1,        #if word frequency < 5, does not train 
    sg=0                # skip-gram (sg=0 CBOW | sg=1 Skip-gram)
)

model.wv.vectors.shape 

import numpy as np

# 1. Choose a secret word
secret_word = "apple"

# 2. Pre-calculate the Top 1000 most similar words (for the "Hot" ranking)
# This list is sorted from most similar (0) to least similar (999)
top_1000_list = [word for word, sim in model.wv.most_similar(secret_word, topn=1000)]

def get_semantle_stats(guess):
    # Check if word exists in our model's vocabulary
    if guess not in model.wv:
        return None, "Word not found in dictionary."

    # A. Calculate Cosine Similarity (-1 to 1)
    # We multiply by 100 to make it look like a score
    similarity = model.wv.similarity(guess, secret_word)
    score = round(similarity * 100, 2)

    # B. Calculate Rank
    if guess == secret_word:
        rank = "Found!"
    elif guess in top_1000_list:
        # Index 0 is the closest word, so we invert it (1000/1000 is closest)
        rank = f"{1000 - top_1000_list.index(guess)}/1000"
    else:
        rank = "Cold"

    return score, rank

(1998, 100)

In [ ]:
print(f"--- Welcome to Semantle (Vocabulary Size: {len(model.wv)}) ---")

while True:
    user_guess = input("Enter your guess: ").strip().lower()
    
    if user_guess == "exit":
        break
        
    score, rank = get_semantle_stats(user_guess)
    
    if score is None:
        print(rank) # Error message
    else:
        print(f"Similarity: {score} | Rank: {rank}")
        
    if rank == "Found!":
        print("Congratulations! You found the secret word.")
        break